# Trenowanie modelu rozpoznawania ruchu



In [ ]:
from pathlib import Path
import sys
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import LeaveOneGroupOut

PROJECT_ROOT = Path.cwd().parent
if not (PROJECT_ROOT / 'data').exists():
    # Działa także, gdy notebook zostanie uruchomiony z głównego folderu projektu.
    PROJECT_ROOT = Path.cwd()

sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))
from imu_pipeline import fuse_sensor_features, read_ximu_csv, sampling_rate_hz
from build_training_dataset import FEATURE_SIGNALS, window_features

MANIFEST_PATH = PROJECT_ROOT / 'data' / 'manifest.csv'
ANNOTATIONS_PATH = PROJECT_ROOT / 'data' / 'annotations.csv'
DATASET_PATH = PROJECT_ROOT / 'data' / 'training_windows.csv'
MODEL_PATH = PROJECT_ROOT / 'models' / 'random_forest_stimming.joblib'
REPORTS_DIR = PROJECT_ROOT / 'reports'
print('Folder projektu:', PROJECT_ROOT)

In [ ]:
manifest = pd.read_csv(MANIFEST_PATH)
annotations = pd.read_csv(ANNOTATIONS_PATH)

required_manifest = {'participant_id', 'session_id', 'wrist_path', 'lumbar_path', 'fully_annotated'}
required_annotations = {'session_id', 'start_s', 'end_s', 'label'}
assert not (required_manifest - set(manifest.columns)), 'Brakuje kolumn w manifest.csv'
assert not (required_annotations - set(annotations.columns)), 'Brakuje kolumn w annotations.csv'
assert manifest['participant_id'].nunique() >= 3, 'Potrzebne są dane co najmniej 3 osób.'

print('Liczba sesji:', len(manifest))
print('Liczba osób:', manifest['participant_id'].nunique())
display(manifest[['participant_id', 'session_id', 'lumbar_time_offset_s']])
display(annotations['label'].value_counts().rename_axis('etykieta').to_frame('liczba_przedziałów'))

In [ ]:
WINDOW_SECONDS = 3.0
HOP_SECONDS = 1.0
rows = []

for _, session in manifest.iterrows():
    wrist_path = PROJECT_ROOT / session['wrist_path']
    lumbar_path = PROJECT_ROOT / session['lumbar_path']
    if not wrist_path.exists() or not lumbar_path.exists():
        raise FileNotFoundError(f"Brak pliku w sesji {session['session_id']}")

    wrist = read_ximu_csv(wrist_path.read_bytes())
    lumbar = read_ximu_csv(lumbar_path.read_bytes())
    offset = float(session.get('lumbar_time_offset_s', 0.0))
    fused = fuse_sensor_features(wrist, lumbar, lumbar_time_offset_s=offset)
    session_labels = annotations[annotations['session_id'] == session['session_id']]
    rows.extend(window_features(fused, session, session_labels, WINDOW_SECONDS, HOP_SECONDS))

dataset = pd.DataFrame(rows)
dataset = dataset[dataset['label'] != 'unknown'].reset_index(drop=True)
DATASET_PATH.parent.mkdir(exist_ok=True)
dataset.to_csv(DATASET_PATH, index=False)
print(f'Zapisano {len(dataset)} okien: {DATASET_PATH}')
display(dataset['label'].value_counts().rename_axis('klasa').to_frame('liczba_okien'))

In [ ]:
METADATA_COLUMNS = {'participant_id', 'session_id', 'window_start_s', 'window_end_s', 'label'}
feature_columns = [c for c in dataset.columns if c not in METADATA_COLUMNS]
X = dataset[feature_columns]
y = dataset['label']
groups = dataset['participant_id']

print('Liczba cech:', len(feature_columns))
print('Liczba okien:', len(X))
print('Liczba osób w walidacji:', groups.nunique())
display(pd.DataFrame({'cecha': feature_columns}))

In [ ]:
def make_model():
    return RandomForestClassifier(
        n_estimators=400,
        class_weight='balanced_subsample',
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=42,
    )

splitter = LeaveOneGroupOut()
predictions = pd.Series(index=dataset.index, dtype='object')

for round_no, (train_idx, test_idx) in enumerate(splitter.split(X, y, groups), start=1):
    tested_person = groups.iloc[test_idx].iloc[0]
    print(f'Runda {round_no}/{groups.nunique()}: test dla {tested_person}')
    model = make_model()
    model.fit(X.iloc[train_idx], y.iloc[train_idx])
    predictions.iloc[test_idx] = model.predict(X.iloc[test_idx])

accuracy = accuracy_score(y, predictions)
report = pd.DataFrame(classification_report(y, predictions, output_dict=True, zero_division=0)).transpose()
print(f'Accuracy: {accuracy:.4f}')
display(report[['precision', 'recall', 'f1-score', 'support']])

In [ ]:
classes = sorted(y.unique())
matrix = pd.DataFrame(confusion_matrix(y, predictions, labels=classes), index=classes, columns=classes)
matrix.index.name = 'prawdziwa_klasa'
matrix.columns.name = 'przewidziana_klasa'

REPORTS_DIR.mkdir(exist_ok=True)
report.to_csv(REPORTS_DIR / 'classification_report.csv')
matrix.to_csv(REPORTS_DIR / 'confusion_matrix.csv')
pd.DataFrame({
    'session_id': dataset['session_id'],
    'participant_id': groups,
    'window_start_s': dataset['window_start_s'],
    'window_end_s': dataset['window_end_s'],
    'actual_label': y,
    'predicted_label': predictions,
}).to_csv(REPORTS_DIR / 'out_of_sample_predictions.csv', index=False)

display(matrix)
print('Zapisano raporty w:', REPORTS_DIR)

In [ ]:
final_model = make_model()
final_model.fit(X, y)
MODEL_PATH.parent.mkdir(exist_ok=True)
joblib.dump({
    'model': final_model,
    'feature_columns': feature_columns,
    'classes': classes,
}, MODEL_PATH)
print('Zapisano model:', MODEL_PATH)